# Step 1.1 - Normalize OpenPhenom embeddings against DMSO controls

**What this notebook does.** It corrects the OpenPhenom embeddings (Step 0.4) for technical variation,
using the DMSO control wells as the reference, and saves "sphered + MAD-robustized" versions.

**Why.** Image features change with the plate, batch, staining intensity and microscope, not only with
the compound. These technical effects can be larger than the biological ones. The standard remedy in
image-based profiling is to express every well *relative to negative-control wells* - here, wells
treated only with DMSO, the solvent, which should look like untreated cells. Two operations are applied:

1. **MAD robustization (robust z-scoring).** For each feature, subtract the median of the DMSO wells
   and divide by their *median absolute deviation* (MAD). A value of 3 then means "three robust
   standard deviations away from untreated cells". Medians are used instead of means because they are
   not thrown off by a few outlier wells.
2. **Sphering (ZCA whitening).** Many features are correlated and carry overlapping information.
   Sphering rotates and rescales the feature space so that, among DMSO wells, all features are
   uncorrelated with equal variance. Afterwards every independent direction of variation counts
   equally, and the plain (Euclidean) distance from DMSO becomes a sensible measure of a compound's
   effect. A small constant (1e-4) is added to the eigenvalues to avoid dividing by near-zero values.

If only one DMSO row exists (e.g. data already averaged per compound), the scale and correlation
structure are estimated from all wells instead.

**Inputs.** `openphenom_384.parquet`, `openphenom_1920.parquet` (Step 0.4).

**Outputs.** `openphenom_384_sphered_mad_robustized.parquet`,
`openphenom_1920_sphered_mad_robustized.parquet`.

> ⚠️ **Version note.** The project history (update of 09/23/2026) reports that errors were found in the
> normalization and that all morphological datasets were re-normalized (files named
> `*_0922_normed.parquet`). The code that produced those files is **not** in this repository. This
> notebook contains the earlier method, whose outputs (`*_sphered_mad_robustized.parquet`) are the
> files used by Steps 1.2, 3 and 4.

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
stages temporary files under `/content/` (Colab's working disk). You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [1]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import median_abs_deviation

### Shared constants
The JUMP identifier of DMSO and the regularization added to eigenvalues during sphering.

In [2]:
# The universal JUMP-CP identifier for DMSO
DMSO_JCP_ID = "JCP2022_033924"

# Sphering regularization parameter (prevents math crashes on highly correlated features)
SPHERING_REGULARIZATION = 1e-4

## Part 1 - Sanity checks: are DMSO control wells present?

Normalization is impossible without control wells, so we first look inside the raw 384-dimensional file.

### 1.1 Which columns are embeddings and which are metadata?

In [3]:
parquet_file = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'
df_test = pd.read_parquet(parquet_file)

num_cols = df_test.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = df_test.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"📊 Total Columns: {len(df_test.columns)}")
print(f"🔢 Numeric Columns Found: {len(num_cols)} -> {num_cols[:5]}")
print(f"🔤 String/Object Columns Found: {len(obj_cols)} -> {obj_cols[:10]}")

### 1.2 How many DMSO rows are there?
Counts rows carrying the DMSO identifier `JCP2022_033924`, or text labels such as "negcon" or "DMSO".

In [4]:
# The OpenPhenom parquet file
parquet_file = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'

print("📂 Loading metadata from OpenPhenom Parquet...")

# Read the file
df = pd.read_parquet(parquet_file)

# 1. Check using the universal JUMP-CP DMSO identifier
dmso_id = 'JCP2022_033924'
if 'Metadata_JCP2022' in df.columns:
    jcp_count = (df['Metadata_JCP2022'].astype(str).str.strip() == dmso_id).sum()
    print(f"✅ Found {jcp_count} row(s) matching '{dmso_id}' in 'Metadata_JCP2022'.")
else:
    print("❌ Column 'Metadata_JCP2022' not found.")

# 2. Check if there are any explicitly labeled "negcon" or "DMSO" rows in other metadata columns
meta_cols = df.select_dtypes(exclude=['number']).columns

print("\n🔍 Scanning all other metadata columns for 'negcon' or 'DMSO'...")
found_alt_labels = False

for col in meta_cols:
    negcon_count = df[col].astype(str).str.contains('negcon', case=False, na=False).sum()
    dmso_count = df[col].astype(str).str.contains('dmso', case=False, na=False).sum()

    if negcon_count > 0:
        print(f"   -> Found {negcon_count} row(s) with 'negcon' in '{col}'")
        found_alt_labels = True
    if dmso_count > 0:
        print(f"   -> Found {dmso_count} row(s) with 'DMSO' in '{col}'")
        found_alt_labels = True

if not found_alt_labels:
    print("   -> No explicit 'negcon' or 'DMSO' text labels found in any other columns.")

print("\n📊 Total rows in the entire dataset:", f"{len(df):,}")

## Part 2 - Normalize the 384-dimensional embeddings

### 2.1 Configuration

In [5]:
# The Parquet file you just assembled
input_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'

# Output normalized file
output_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized.parquet'

### 2.2 Load the embeddings
Numeric columns are the embedding; text columns are metadata.

In [6]:
print("📂 Loading OpenPhenom embeddings from GCS...")
start_time = time.time()
df = pd.read_parquet(input_parquet)
print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns in {time.time() - start_time:.1f}s.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"   -> Found {len(meta_cols)} metadata columns and {len(emb_cols)} embedding dimensions.")

### 2.3 Find the DMSO control wells
Every metadata column is searched for the DMSO identifier `JCP2022_033924`.

In [7]:
print(f"\n🔍 Searching metadata for DMSO identifier: '{DMSO_JCP_ID}'...")

# Create a blank mask to track DMSO rows
dmso_mask = pd.Series(False, index=df.index)

# Scan all text columns for the JCP identifier
for col in meta_cols:
    match = df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper()
    if match.sum() > 0:
        print(f"   -> Found DMSO matches in column: '{col}'")
        dmso_mask = dmso_mask | match

dmso_count = dmso_mask.sum()

print("\n" + "="*50)
print(f"📊 DMSO COUNT RESULT: {dmso_count:,} wells identified.")
print("="*50)

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_JCP_ID}' anywhere in the metadata.")

# Separate the embedding arrays
dmso_embs = df.loc[dmso_mask, emb_cols].values
all_embs = df[emb_cols].values

### 2.4 Step A - MAD robustization
Center each feature on the DMSO median and divide by the DMSO MAD. Features that barely vary among DMSO wells (MAD ≤ 1e-5) are dropped because they cannot be rescaled.

In [8]:
print("\n📐 Step A: MAD-Robustizing (Centering and Scaling)...")
start_math = time.time()

# Center the data exactly on the DMSO median
center_median = np.median(dmso_embs, axis=0)

# 🚨 MATHEMATICAL SAFEGUARD:
# If there is only 1 DMSO row (it was aggregated), its variance is 0.
# We must use the variance of the entire dataset to scale the features instead!
if dmso_count > 1:
    print("   -> Multiple DMSO rows detected. Scaling by DMSO variance (MAD).")
    spread_mad = median_abs_deviation(dmso_embs, axis=0)
else:
    print("   -> Only 1 DMSO row detected. Scaling by Global Dataset variance (MAD).")
    spread_mad = median_abs_deviation(all_embs, axis=0)

# Drop "Dead Neurons" (Features with zero variance) to prevent division by zero
valid_features = spread_mad > 1e-5
dropped_count = len(emb_cols) - np.sum(valid_features)
print(f"   -> Dropping {dropped_count} 'dead' features with ~0 variance.")

valid_emb_cols = np.array(emb_cols)[valid_features]
all_embs_clean = all_embs[:, valid_features]

center_median_clean = center_median[valid_features]
spread_mad_clean = spread_mad[valid_features]

# Apply Robust Z-Scoring to the entire dataset
all_robust = (all_embs_clean - center_median_clean) / spread_mad_clean

# If we have enough DMSO, we also need the robustized DMSO array for sphering
if dmso_count > 1:
    dmso_embs_clean = dmso_embs[:, valid_features]
    dmso_robust = (dmso_embs_clean - center_median_clean) / spread_mad_clean

### 2.5 Step B - sphering (ZCA whitening)
Compute the covariance of the robustized DMSO wells, take its eigendecomposition and build the whitening matrix `W = E · D^(-1/2) · Eᵀ`; every well is multiplied by `W`.

In [9]:
print("🔮 Step B: Sphering (ZCA Whitening) to remove feature correlations...")

if dmso_count > 1:
    print("   -> Calculating covariance matrix from DMSO controls...")
    cov_matrix = np.cov(dmso_robust, rowvar=False)
else:
    print("   -> Single DMSO row detected. Calculating covariance matrix from Global Dataset...")
    cov_matrix = np.cov(all_robust, rowvar=False)

# Eigendecomposition of the covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Clip tiny negative eigenvalues caused by floating point precision limits
eigenvalues = np.clip(eigenvalues, a_min=0, a_max=None)

# Calculate the ZCA Sphering Matrix: W = E * D^(-1/2) * E.T
D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues + SPHERING_REGULARIZATION))
W_zca = eigenvectors @ D_inv_sqrt @ eigenvectors.T

# Apply the sphering transformation to the ENTIRE dataset
all_sphered = all_robust @ W_zca

print(f"   ✅ Mathematics complete in {time.time() - start_math:.2f} seconds.")

### 2.6 Re-attach metadata and save
Embedding columns are renamed `OpenPhenom_Sphered_Dim_1 …`.

In [10]:
print("\n💾 Reassembling DataFrame and saving to Google Cloud Storage...")

# Create new dimension column names to reflect the transformation
new_dim_cols = [f"OpenPhenom_Sphered_Dim_{i+1}" for i in range(all_sphered.shape[1])]

# Build the final dataframe
df_sphered_features = pd.DataFrame(all_sphered, columns=new_dim_cols, index=df.index)

# Re-attach the metadata
df_final = pd.concat([df[meta_cols], df_sphered_features], axis=1)

# Ensure string metadata stays as strings for PyArrow compatibility
for col in meta_cols:
    df_final[col] = df_final[col].astype(str)

# Save to Parquet natively
df_final.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! Sphered embeddings saved permanently to:\n   {output_parquet}")

# Verify output
print("\n" + "-" * 75)
print(f"📊 Final Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print(f"📌 Index Name:  {df_final.index.name}")
print("-" * 75)
display(df_final.head(3))

## Part 3 - Normalize the 1,920-dimensional embeddings
The identical procedure.

### 3.1 Configuration

In [11]:
# The Parquet file you just assembled
input_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920.parquet'

# Output normalized file
output_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized.parquet'

### 3.2 Load the embeddings
Numeric columns are the embedding; text columns are metadata.

In [12]:
print("📂 Loading OpenPhenom embeddings from GCS...")
start_time = time.time()
df = pd.read_parquet(input_parquet)
print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns in {time.time() - start_time:.1f}s.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"   -> Found {len(meta_cols)} metadata columns and {len(emb_cols)} embedding dimensions.")

### 3.3 Find the DMSO control wells
Every metadata column is searched for the DMSO identifier `JCP2022_033924`.

In [13]:
print(f"\n🔍 Searching metadata for DMSO identifier: '{DMSO_JCP_ID}'...")

# Create a blank mask to track DMSO rows
dmso_mask = pd.Series(False, index=df.index)

# Scan all text columns for the JCP identifier
for col in meta_cols:
    match = df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper()
    if match.sum() > 0:
        print(f"   -> Found DMSO matches in column: '{col}'")
        dmso_mask = dmso_mask | match

dmso_count = dmso_mask.sum()

print("\n" + "="*50)
print(f"📊 DMSO COUNT RESULT: {dmso_count:,} wells identified.")
print("="*50)

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_JCP_ID}' anywhere in the metadata.")

# Separate the embedding arrays
dmso_embs = df.loc[dmso_mask, emb_cols].values
all_embs = df[emb_cols].values

### 3.4 Step A - MAD robustization
Center each feature on the DMSO median and divide by the DMSO MAD. Features that barely vary among DMSO wells (MAD ≤ 1e-5) are dropped because they cannot be rescaled.

In [14]:
print("\n📐 Step A: MAD-Robustizing (Centering and Scaling)...")
start_math = time.time()

# Center the data exactly on the DMSO median
center_median = np.median(dmso_embs, axis=0)

# 🚨 MATHEMATICAL SAFEGUARD:
# If there is only 1 DMSO row (it was aggregated), its variance is 0.
# We must use the variance of the entire dataset to scale the features instead!
if dmso_count > 1:
    print("   -> Multiple DMSO rows detected. Scaling by DMSO variance (MAD).")
    spread_mad = median_abs_deviation(dmso_embs, axis=0)
else:
    print("   -> Only 1 DMSO row detected. Scaling by Global Dataset variance (MAD).")
    spread_mad = median_abs_deviation(all_embs, axis=0)

# Drop "Dead Neurons" (Features with zero variance) to prevent division by zero
valid_features = spread_mad > 1e-5
dropped_count = len(emb_cols) - np.sum(valid_features)
print(f"   -> Dropping {dropped_count} 'dead' features with ~0 variance.")

valid_emb_cols = np.array(emb_cols)[valid_features]
all_embs_clean = all_embs[:, valid_features]

center_median_clean = center_median[valid_features]
spread_mad_clean = spread_mad[valid_features]

# Apply Robust Z-Scoring to the entire dataset
all_robust = (all_embs_clean - center_median_clean) / spread_mad_clean

# If we have enough DMSO, we also need the robustized DMSO array for sphering
if dmso_count > 1:
    dmso_embs_clean = dmso_embs[:, valid_features]
    dmso_robust = (dmso_embs_clean - center_median_clean) / spread_mad_clean

### 3.5 Step B - sphering (ZCA whitening)
Compute the covariance of the robustized DMSO wells, take its eigendecomposition and build the whitening matrix `W = E · D^(-1/2) · Eᵀ`; every well is multiplied by `W`.

In [15]:
print("🔮 Step B: Sphering (ZCA Whitening) to remove feature correlations...")

if dmso_count > 1:
    print("   -> Calculating covariance matrix from DMSO controls...")
    cov_matrix = np.cov(dmso_robust, rowvar=False)
else:
    print("   -> Single DMSO row detected. Calculating covariance matrix from Global Dataset...")
    cov_matrix = np.cov(all_robust, rowvar=False)

# Eigendecomposition of the covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Clip tiny negative eigenvalues caused by floating point precision limits
eigenvalues = np.clip(eigenvalues, a_min=0, a_max=None)

# Calculate the ZCA Sphering Matrix: W = E * D^(-1/2) * E.T
D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues + SPHERING_REGULARIZATION))
W_zca = eigenvectors @ D_inv_sqrt @ eigenvectors.T

# Apply the sphering transformation to the ENTIRE dataset
all_sphered = all_robust @ W_zca

print(f"   ✅ Mathematics complete in {time.time() - start_math:.2f} seconds.")

### 3.6 Re-attach metadata and save
Embedding columns are renamed `OpenPhenom_Sphered_Dim_1 …`.

In [16]:
print("\n💾 Reassembling DataFrame and saving to Google Cloud Storage...")

# Create new dimension column names to reflect the transformation
new_dim_cols = [f"OpenPhenom_Sphered_Dim_{i+1}" for i in range(all_sphered.shape[1])]

# Build the final dataframe
df_sphered_features = pd.DataFrame(all_sphered, columns=new_dim_cols, index=df.index)

# Re-attach the metadata
df_final = pd.concat([df[meta_cols], df_sphered_features], axis=1)

# Ensure string metadata stays as strings for PyArrow compatibility
for col in meta_cols:
    df_final[col] = df_final[col].astype(str)

# Save to Parquet natively
df_final.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! Sphered embeddings saved permanently to:\n   {output_parquet}")

# Verify output
print("\n" + "-" * 75)
print(f"📊 Final Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print(f"📌 Index Name:  {df_final.index.name}")
print("-" * 75)
display(df_final.head(3))